In [ ]:
import os
import csv
import cv2
import glob
import numpy as np
import mediapipe as mp
from dataclasses import dataclass
from typing import Optional, List, Union, Dict

In [ ]:
def listar_arquivos(pasta: str):
    if not os.path.exists(pasta):
        print(f"A pasta '{pasta}' não existe.")
        return

    arquivos = [f for f in os.listdir(pasta) if os.path.isfile(os.path.join(pasta, f))]
    lista_arquivos = []
    for arquivo in arquivos:
        lista_arquivos.append(arquivo)
    if lista_arquivos:
        return lista_arquivos
    else:
        print(f"Nenhum arquivo encontrado na pasta '{pasta}'.")

In [ ]:
# =========================
# Config
# =========================
@dataclass
class Config:
    local_arquivo: str = r"./videos"   # arquivo ou pasta
    saida_dir: str = r"./saida"

    # pipeline_stage:
    # "preprocess" -> só pré-processamento
    # "segment"    -> pré-processamento + segmentação
    # "pose"       -> pré-processamento + (segmentação opcional) + pose
    pipeline_stage: str = "pose"

    segmentation_method: str = "mediapipe"
    use_segmentation_in_pose: bool = True  # se False, roda pose direto no frame preprocessado

    # Pose: rodar no frame original preprocessado ("original") ou no recortado ("masked")
    pose_input: str = "original"  # "original" ou "masked"

    # Pré-processamento
    max_width: int = 640
    use_clahe: bool = True
    use_denoise: bool = False
    denoise_h: int = 5
    use_unsharp: bool = True

    # Segmentação mediapipe
    mp_seg_model_selection: int = 0
    mp_seg_threshold: float = 0.10

    # Pós-processamento da máscara
    morph_kernel: int = 5
    keep_largest_component: bool = True

    # Preview em tempo real
    preview: bool = False
    preview_only_first_seconds: float = 5.0  # 0 = mostra sempre
    preview_window_name: str = "Preview"

    # Saídas
    write_csv: bool = True
    draw_segmentation_overlay: bool = True
    overlay_alpha: float = 0.35

    # Seleção de landmarks a salvar no CSV:
    # - None (default): salva todos os 33
    # - lista de indices: [0, 11, 12, ...]
    # - lista de nomes: ["LEFT_SHOULDER", "RIGHT_SHOULDER", ...]
    selected_landmarks: Optional[List[Union[int, str]]] = None

In [ ]:
# =========================
# Utilidades
# =========================
VIDEO_EXTS = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v")

def pick_video(path_or_dir: str) -> str:
    if os.path.isfile(path_or_dir):
        return path_or_dir
    if not os.path.isdir(path_or_dir):
        raise FileNotFoundError(f"local_arquivo não é arquivo nem pasta: {path_or_dir}")

    files: List[str] = []
    for ext in VIDEO_EXTS:
        files += glob.glob(os.path.join(path_or_dir, f"*{ext}"))
        files += glob.glob(os.path.join(path_or_dir, f"*{ext.upper()}"))

    if not files:
        raise FileNotFoundError(f"Nenhum vídeo encontrado em: {path_or_dir}")
    files.sort()
    return files[0]


def ensure_dir(d: str) -> None:
    os.makedirs(d, exist_ok=True)


def resize_keep_aspect(frame: np.ndarray, max_width: int) -> np.ndarray:
    h, w = frame.shape[:2]
    if w <= max_width:
        return frame
    scale = max_width / float(w)
    new_w = max_width
    new_h = int(round(h * scale))
    return cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)


def apply_clahe_bgr(frame_bgr: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l2 = clahe.apply(l)
    lab2 = cv2.merge([l2, a, b])
    return cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)


def denoise_frame(frame_bgr: np.ndarray, h: int) -> np.ndarray:
    return cv2.fastNlMeansDenoisingColored(frame_bgr, None, h, h, 7, 21)


def unsharp_mask(frame_bgr: np.ndarray) -> np.ndarray:
    blur = cv2.GaussianBlur(frame_bgr, (0, 0), 1.0)
    sharp = cv2.addWeighted(frame_bgr, 1.4, blur, -0.4, 0)
    return sharp


def preprocess(frame_bgr: np.ndarray, cfg: Config) -> np.ndarray:
    x = resize_keep_aspect(frame_bgr, cfg.max_width)
    if cfg.use_clahe:
        x = apply_clahe_bgr(x)
    if cfg.use_denoise:
        x = denoise_frame(x, cfg.denoise_h)
    if cfg.use_unsharp:
        x = unsharp_mask(x)
    return x


def postprocess_mask(mask01: np.ndarray, cfg: Config) -> np.ndarray:
    mask = (mask01 * 255).astype(np.uint8)
    thr = int(np.clip(cfg.mp_seg_threshold * 255, 0, 255))
    _, binm = cv2.threshold(mask, thr, 255, cv2.THRESH_BINARY)

    k = max(3, cfg.morph_kernel | 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))

    binm = cv2.morphologyEx(binm, cv2.MORPH_OPEN, kernel, iterations=1)
    binm = cv2.morphologyEx(binm, cv2.MORPH_CLOSE, kernel, iterations=2)

    if cfg.keep_largest_component:
        num, labels, stats, _ = cv2.connectedComponentsWithStats(binm, connectivity=8)
        if num > 1:
            largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
            binm = np.where(labels == largest, 255, 0).astype(np.uint8)

    return binm


def apply_mask(frame_bgr: np.ndarray, mask255: np.ndarray) -> np.ndarray:
    return cv2.bitwise_and(frame_bgr, frame_bgr, mask=mask255)

In [ ]:
# =========================
# Landmarks: índices e nomes
# =========================
def get_landmark_name_map() -> Dict[str, int]:
    # mp.solutions.pose.PoseLandmark é um enum com 33 itens
    PoseLandmark = mp.solutions.pose.PoseLandmark
    return {name: PoseLandmark[name].value for name in PoseLandmark.__members__.keys()}

def normalize_selected_landmarks(selected: Optional[List[Union[int, str]]]) -> Optional[List[int]]:
    if selected is None:
        return None
    name_map = get_landmark_name_map()
    out: List[int] = []
    for item in selected:
        if isinstance(item, int):
            if not (0 <= item <= 32):
                raise ValueError(f"Landmark index inválido: {item}. Deve estar entre 0 e 32.")
            out.append(item)
        elif isinstance(item, str):
            key = item.strip().upper()
            if key not in name_map:
                raise ValueError(f"Landmark name inválido: {item}. Use nomes como 'LEFT_SHOULDER'.")
            out.append(name_map[key])
        else:
            raise TypeError(f"selected_landmarks aceita int ou str; recebido: {type(item)}")
    # remove duplicados mantendo ordem
    seen = set()
    uniq = []
    for i in out:
        if i not in seen:
            seen.add(i)
            uniq.append(i)
    return uniq

In [ ]:
# =========================
# Main
# =========================
def main(cfg: Config) -> None:
    video_path = pick_video(cfg.local_arquivo)
    ensure_dir(cfg.saida_dir)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir o vídeo: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:
        fps = 30.0

    base = os.path.splitext(os.path.basename(video_path))[0]
    out_video_path = os.path.join(cfg.saida_dir, f"videos/{base}_{cfg.pipeline_stage}.mp4")
    out_csv_path = os.path.join(cfg.saida_dir, f"landmarks/{base}_landmarks.csv")

    stage = cfg.pipeline_stage.lower().strip()
    if stage not in ("preprocess", "segment", "pose"):
        raise ValueError("pipeline_stage deve ser: 'preprocess', 'segment' ou 'pose'")

    # MediaPipe
    mp_pose = mp.solutions.pose
    mp_draw = mp.solutions.drawing_utils
    mp_styles = mp.solutions.drawing_styles

    selfie_seg = None
    if stage in ("segment", "pose") and cfg.segmentation_method.lower() == "mediapipe":
        mp_selfie = mp.solutions.selfie_segmentation
        selfie_seg = mp_selfie.SelfieSegmentation(model_selection=cfg.mp_seg_model_selection)

    pose = None
    if stage == "pose":
        pose = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )

    # Landmarks selecionados
    selected_idx = normalize_selected_landmarks(cfg.selected_landmarks)
    if selected_idx is None:
        selected_idx = list(range(33))

    # CSV
    csv_file = None
    csv_writer = None
    if stage == "pose" and cfg.write_csv:
        csv_file = open(out_csv_path, "w", newline="", encoding="utf-8")
        csv_writer = csv.writer(csv_file)
        header = ["frame", "time_sec"]
        for i in selected_idx:
            header += [f"lm{i}_x", f"lm{i}_y", f"lm{i}_z", f"lm{i}_vis"]
        csv_writer.writerow(header)

    writer = None
    frame_idx = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            t = frame_idx / float(fps)
            frame_pp = preprocess(frame, cfg)

            # =========================
            # STAGE 1: preprocess-only
            # =========================
            if stage == "preprocess":
                vis = frame_pp.copy()

            # =========================
            # STAGE 2/3: segment
            # =========================
            else:
                # segmentação humano vs fundo
                if cfg.segmentation_method.lower() == "mediapipe":
                    rgb = cv2.cvtColor(frame_pp, cv2.COLOR_BGR2RGB)
                    seg = selfie_seg.process(rgb)
                    mask255 = postprocess_mask(seg.segmentation_mask, cfg)
                else:
                    mask255 = mask255

                masked = apply_mask(frame_pp, mask255)

                if stage == "segment":
                    vis = frame_pp.copy()
                    if cfg.draw_segmentation_overlay:
                        color_mask = np.zeros_like(vis)
                        color_mask[:, :, 1] = mask255  # verde
                        vis = cv2.addWeighted(vis, 1.0, color_mask, cfg.overlay_alpha, 0)
                else:
                    # =========================
                    # STAGE 3: pose
                    # =========================
                    if cfg.use_segmentation_in_pose and cfg.pose_input.lower() == "masked":
                        pose_in = masked
                    else:
                        pose_in = frame_pp

                    pose_rgb = cv2.cvtColor(pose_in, cv2.COLOR_BGR2RGB)
                    res = pose.process(pose_rgb)

                    vis = frame_pp.copy()
                    if cfg.draw_segmentation_overlay and cfg.use_segmentation_in_pose:
                        color_mask = np.zeros_like(vis)
                        color_mask[:, :, 1] = mask255
                        vis = cv2.addWeighted(vis, 1.0, color_mask, cfg.overlay_alpha, 0)

                    if res.pose_landmarks:
                        mp_draw.draw_landmarks(
                            vis,
                            res.pose_landmarks,
                            mp_pose.POSE_CONNECTIONS,
                            landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style()
                        )

                    # CSV (apenas landmarks selecionados)
                    if cfg.write_csv and csv_writer is not None:
                        row = [frame_idx, t]
                        if res.pose_landmarks:
                            lms = res.pose_landmarks.landmark
                            for i in selected_idx:
                                row += [lms[i].x, lms[i].y, lms[i].z, lms[i].visibility]
                        else:
                            row += [np.nan] * (len(selected_idx) * 4)
                        csv_writer.writerow(row)

            # texto informativo
            # cv2.putText(
            #     vis,
            #     f"{base} | stage={stage} | frame={frame_idx} | t={t:.2f}s",
            #     (10, 30),
            #     cv2.FONT_HERSHEY_SIMPLEX,
            #     0.7,
            #     (255, 255, 255),
            #     2,
            #     cv2.LINE_AA,
            # )

            # Preview (só primeiros N segundos, se configurado)
            if cfg.preview:
                if cfg.preview_only_first_seconds <= 0 or t <= cfg.preview_only_first_seconds:
                    cv2.imshow(cfg.preview_window_name, vis)
                    key = cv2.waitKey(10) & 0xFF
                    if key in (27, ord('q')):
                        break

            # inicializa writer no 1º frame
            if writer is None:
                h, w = vis.shape[:2]
                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                writer = cv2.VideoWriter(out_video_path, fourcc, fps, (w, h))

            writer.write(vis)
            frame_idx += 1

    finally:
        cap.release()
        if writer is not None:
            writer.release()
        if pose is not None:
            pose.close()
        if selfie_seg is not None:
            selfie_seg.close()
        if csv_file is not None:
            csv_file.close()
        if cfg.preview:
            cv2.destroyAllWindows()

    print("OK!")
    print("Vídeo:", out_video_path)
    if stage == "pose" and cfg.write_csv:
        print("CSV :", out_csv_path)

In [ ]:
if __name__ == "__main__":
    lista_arquivos = listar_arquivos("./entrada")

    for arquivo in lista_arquivos:
        cfg = Config(
            pipeline_stage="pose", # preprocess, segment ou pose
            local_arquivo=r"./entrada"+"/"+arquivo,
            segmentation_method="mediapipe",
            use_denoise=False,
            max_width=640,
            use_segmentation_in_pose=True,
            pose_input="original",
            selected_landmarks=["LEFT_HIP","RIGHT_HIP","LEFT_SHOULDER","RIGHT_SHOULDER","LEFT_ELBOW","RIGHT_ELBOW","LEFT_WRIST","RIGHT_WRIST"],
            preview=False
        )

        main(cfg)